Single-cell NicheNet’s ligand activity analysis
================


This vignette shows how NicheNet can be used to predict which ligands
might be active in single-cells. If a ligand has a high activity in a
cell, this means that target genes of that ligand are stronger expressed
in that cell than in other cells. In this example, we will use data from
Puram et al. to explore intercellular communication in the tumor
microenvironment in head and neck squamous cell carcinoma (HNSCC) (See
Puram et al. 2017). More specifically, we will assess the activity of
cancer-associated fibroblast (CAF) ligands in malignant cells. 

In order to prioritize ligands regulating a process of interest, you can
perform a regression/correlation analysis between ligand activities in
cells, and scores of a cell corresponding to the process of interest.
For example, in this case study we were interested in finding ligands
regulating p-EMT. Therefore we correlated ligand activities to the p-EMT
scores of cells.

The purpose of this single-cell ligand activity analysis is to offer a
complementary way to prioritize ligands driving the process of interest
and to better analyze heterogeneity in ligand activity between different
cells.

In [ ]:
from nichenetpy.utils import read_csv_rows
from nichenetpy.gene_symbol import human_alias_info
from nichenetpy.io import read_csc_matrix
from nichenetpy.utils import subset_matrix
from nichenetpy.normalization import scale_quantile

from math import log

import pickle
import os
import requests
import numpy as np

Download the model files

In [ ]:
model_path = os.path.normpath("./tutorial_files/model/human")
if not os.path.exists(model_path):
    os.makedirs(model_path)
filename = "nichenet_human.pkl"
file_path = os.path.join("./tutorial_files", filename)
if not os.path.exists(file_path):
    res = requests.get(f"https://zenodo.org/records/14887637/files/{filename}")
    with open(file_path, "wb") as file:
        file.write(res.content)

Download the HNSCC files

In [ ]:
hnscc_path = os.path.normpath("./tutorial_files/hnscc")
if not os.path.exists(hnscc_path):
    os.makedirs(hnscc_path)
for filename in (
    "expressed_genes.csv",
    "hnscc_expression.bin",
    "pemt_signature.txt",
    "sample_info.csv"
):
    file_path = os.path.join(hnscc_path, filename)
    if not os.path.exists(file_path):
        res = requests.get(f"https://zenodo.org/records/14859451/files/{filename}")
        with open(file_path, "wb") as file:
            file.write(res.content)

In [ ]:
with open("./tutorial_files/nichenet_human.pkl", "rb") as file:
    model = pickle.loads(file.read())
predictor = model["predictor"]
lr_network = model["lr_network"]
lr_sig = model["lr_sig"]

In [ ]:
exp_mat, rows, cols = read_csc_matrix(os.path.join(hnscc_path, "hnscc_expression.bin"))
cols = human_alias_info.alias_to_symbol(cols)
sample_info_col_names, sample_info = read_csv_rows(os.path.join(hnscc_path, "sample_info.csv"))

Determine which genes are expressed in CAFs and malignant cells from high quality primary tumors. Therefore, we wil not consider cells from tumor samples of less quality or from lymph node metastases. To determine expressed genes, we use the definition used by Puram et al.

In [ ]:
tumors_remove = {"HN10","HN","HN12", "HN13", "HN24", "HN7", "HN8","HN23"}
CAF_cells = [e[5] for e in sample_info if e[1] == 0 and e[4] == "CAF" and e[6] not in tumors_remove]
malignant_cells = [e[5] for e in sample_info if e[1] == 0 and e[2] == 1 and e[6] not in tumors_remove]
row2id = dict(zip(rows, range(len(rows))))

def get_exp(mat, cols):
    agg_exp = [
        log(sum((10*(2**x - 1) for x in mat[:, i]))/mat.shape[0] + 1, 2)
        for i in range(mat.shape[1])
    ]
    return {gene for gene, x in zip(cols, agg_exp) if x >= 4}

exp_mat = np.array(exp_mat.todense())
expressed_genes_CAFs = get_exp(subset_matrix(exp_mat, [row2id[cell] for cell in CAF_cells]), cols)
expressed_genes_malignant = get_exp(subset_matrix(exp_mat, [row2id[cell] for cell in malignant_cells]), cols)

### Perform NicheNet’s single-cell ligand activity analysis

In a first step, we will define a set of potentially active ligands. As
potentially active ligands, we will use ligands that are 1) expressed by
CAFs and 2) can bind a (putative) receptor expressed by malignant cells.
Putative ligand-receptor links were gathered from NicheNet’s
ligand-receptor data sources.

In [ ]:
ligands = lr_network.get_ligands()
expressed_ligands = ligands.intersection(expressed_genes_CAFs)
receptors = lr_network.get_receptors()
expressed_receptors = receptors.intersection(expressed_genes_malignant)
potential_ligands = {ligand for ligand, receptor in lr_network if ligand in expressed_ligands and receptor in expressed_receptors}

In a second step, we will scale the single-cell expression data (including only expressed genes).

In [ ]:
background_expressed_genes = expressed_genes_malignant[[e in predictor.row_names for e in expressed_genes_malignant]]
row2id = dict(zip(rows), range(len(rows)))
col2id = dict(zip(cols), range(len(cols)))
expression_scaled = scale_quantile(
    subset_matrix(
        exp_mat,
        rows=[row2id(e) for e in malignant_cells],
        cols=[col2id(e) for e in background_expressed_genes]
    )
)

Now perform the ligand activity analysis: infer how well NicheNet’s ligand-target potential scores can predict whether a gene belongs to most strongly expressed genes in a cell compared to other cells. To reduce the running time for this vignette, we will perform the analysis only on 10 example cells from the HN5 tumor. This vignette’s only purpose is to illustrate the analysis.

In [ ]:
malignant_hn5_ids = [e["tumor" == "HN5" and e["Lymph node"] == 0 and e["classified as cancer cell"] == 1] for e in sample_info]["cell"][:10]
ligand_activities = predict_single_cell_ligand_activities()